## Classifies Culpa Reviews by A 4-quandrant Political Compass

In [ ]:
# Dependencies/libraries
import sys
import torch
import os
import pandas as pd
from transformers import pipeline

In [ ]:
# Check import success
print(sys.version)
print(torch.__version__)

In [ ]:
# This analysis involves a classifier for each axis
# X-axis model
focus_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

# Y-axis model
sentiment_classifier = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest"
)

In [ ]:
# Read in the review sample csv 
cwd = os.getcwd()
csv_path = os.path.join(cwd, 'review.csv')

df = pd.read_csv(csv_path)

# Drop the unecessary information 
df = df.drop(columns = ['5',
                        '3',
                        'Unnamed: 3',
                        'museum project, midterm, final.',
                        '2000-01-01 00:00:00'])

df.loc[-1] = df.columns.to_list()
df.index = df.index + 1

df = df.sort_index()
df = df.rename(columns={df.columns[0]: "review", df.columns[1]: "rating"})
df.head()

In [ ]:
# Classify the focus (prof or course) via zero-shot classification
candidate_labels = [
    "the review evaluates the professor: mastery/knowledge of the material, instructor, teaching style, clarity, personality, helpfulness, or behavior",
    "the review evaluates the course: workload, exams, quizzes, assignments, readings, grading, difficulty, or requirements"
]

def classify_focus(review):
    review = str(review)

    result = focus_classifier(
        review[:1000],
        candidate_labels=candidate_labels,
        hypothesis_template="This student review is mainly about {}."
    )

    scores = dict(zip(result["labels"], result["scores"]))

    professor_prob = scores[candidate_labels[0]]
    course_prob = scores[candidate_labels[1]]

    focus_score = course_prob - professor_prob

    if focus_score < -0.20:
        label = "professor-focused"
    elif focus_score > 0.20:
        label = "course-focused"
    else:
        label = "mixed"

    return pd.Series({
        "professor_prob": professor_prob,
        "course_prob": course_prob,
        "focus_score": focus_score,
        "focus_label": label
    })